# ULTRA-MoCap Full LOSO Training (Google Colab Web)

This notebook runs the full 13-fold LOSO experiment on Colab GPU using the existing training script.

Run order:
1. GPU preflight
2. Mount Drive
3. Install dependencies
4. Resolve repo + dataset paths
5. Launch full LOSO training
6. Sync results to Drive

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('GPU not active. In Colab, go to Runtime > Change runtime type > GPU.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install -q numpy pandas scipy scikit-learn tqdm h5py

In [ ]:
from pathlib import Path
import subprocess

# ---------------- User-configurable defaults ----------------
# Full run: 0 means all 13 folds.
MAX_FOLDS = 0
MODALITIES = 'emg,imu,imu_emg'
RESULT_TAG = 'colab_full_loso'

# EMG training overrides
EMG_MODEL_VARIANT = 'lstm_msa'
EMG_EPOCHS = 50
EMG_PATIENCE = 12
EMG_LR = 5e-4

# Optional: set to 1 only for quick debug runs.
SMOKE = 0

# ---------------- Resolve repository path ----------------
drive_roots = [Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')]
drive_root = next((p for p in drive_roots if p.exists()), None)
assert drive_root is not None, 'Drive root not found. Re-run mount cell.'

repo_candidates = [
    drive_root / 'research-paper',
    drive_root / 'ULTRA-MoCap-Kinematics-Analysis',
    drive_root / 'My Drive' / 'research-paper',
]

def has_training_script(repo_path: Path) -> bool:
    return (repo_path / 'Code-base/MocapDatasetScripting_REALLAB/scripts/training/conv1d_bigru_loso.py').exists()

repo_dir = next((p for p in repo_candidates if has_training_script(p)), None)

if repo_dir is None:
    repo_dir = Path('/content/ULTRA-MoCap-Kinematics-Analysis')
    if not has_training_script(repo_dir):
        print('Cloning repo to /content ...')
        subprocess.run([
            'git', 'clone',
            'https://github.com/MeghVyas3132/ULTRA-MoCap-Kinematics-Analysis.git',
            str(repo_dir),
        ], check=True)

REPO_DIR = str(repo_dir)

# ---------------- Resolve dataset H5 path ----------------
h5_candidates = [
    drive_root / 'research-paper/Dataset/ULTra-MoCap-processed/All_subjects_data.h5',
    drive_root / 'research-paper/Dataset/28751156/ULTra-MoCap-processed/All_subjects_data.h5',
    drive_root / 'Dataset/ULTra-MoCap-processed/All_subjects_data.h5',
    drive_root / 'Dataset/28751156/ULTra-MoCap-processed/All_subjects_data.h5',
    Path(REPO_DIR) / 'Dataset/ULTra-MoCap-processed/All_subjects_data.h5',
    Path(REPO_DIR) / 'Dataset/28751156/ULTra-MoCap-processed/All_subjects_data.h5',
]

h5_path = next((p for p in h5_candidates if p.exists()), None)
if h5_path is None:
    print('All_subjects_data.h5 not found. Checked:')
    for p in h5_candidates:
        print('  -', p)
    raise RuntimeError('Place All_subjects_data.h5 in one of the listed paths and rerun.')

H5_PATH = str(h5_path)

# ---------------- Fast cache/output paths ----------------
USE_LOCAL_SSD_CACHE = True
if USE_LOCAL_SSD_CACHE:
    DATASET_ROOT = '/content/mocap_cache/datasets'
    RESULTS_FOLDER = '/content/mocap_cache/results/Results_ConvBiGRU_colab'
else:
    DATASET_ROOT = f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/datasets'
    RESULTS_FOLDER = f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/results/Results_ConvBiGRU_colab'

RESULTS_ZIP_PATH = f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/results/ConvBiGRU_results_colab_gpu.zip'
DRIVE_RESULTS_DIR = str(drive_root / 'research-paper/Code-base/MocapDatasetScripting_REALLAB/results/Results_ConvBiGRU_colab_gpu')

print('REPO_DIR =', REPO_DIR)
print('H5_PATH =', H5_PATH)
print('DATASET_ROOT =', DATASET_ROOT)
print('RESULTS_FOLDER =', RESULTS_FOLDER)
print('DRIVE_RESULTS_DIR =', DRIVE_RESULTS_DIR)
print('MAX_FOLDS =', MAX_FOLDS)
print('MODALITIES =', MODALITIES)
print('RESULT_TAG =', RESULT_TAG)
print('SMOKE =', SMOKE)

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), (
    'CUDA GPU is not available in this runtime. '
    'Switch Colab runtime to GPU and rerun.'
)
print('Launching on GPU:', torch.cuda.get_device_name(0))

assert Path(REPO_DIR).exists(), f'Repo path not found: {REPO_DIR}'
assert Path(H5_PATH).exists(), f'H5 path not found: {H5_PATH}'

Path(DATASET_ROOT).mkdir(parents=True, exist_ok=True)
Path(RESULTS_FOLDER).mkdir(parents=True, exist_ok=True)
Path(RESULTS_ZIP_PATH).parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env.update({
    'FAST_MODE': '1',
    'MATMUL_PRECISION': 'high',
    'DATA_H5_PATH': H5_PATH,
    'DATASET_ROOT': DATASET_ROOT,
    'RESULTS_FOLDER': RESULTS_FOLDER,
    'RESULTS_ZIP_PATH': RESULTS_ZIP_PATH,
    'RESULT_TAG': RESULT_TAG,
    'MAX_FOLDS': str(MAX_FOLDS),
    'MODALITIES': MODALITIES,
    'DATALOADER_WORKERS': '8',
    'PREFETCH_FACTOR': '4',
    'PERSISTENT_WORKERS': '1',
    'EMG_MODEL_VARIANT': EMG_MODEL_VARIANT,
    'EMG_EPOCHS': str(EMG_EPOCHS),
    'EMG_PATIENCE': str(EMG_PATIENCE),
    'EMG_LR': str(EMG_LR),
})

if int(SMOKE) == 1:
    env['SMOKE'] = '1'

cmd = [
    sys.executable,
    '-u',
    'Code-base/MocapDatasetScripting_REALLAB/scripts/training/conv1d_bigru_loso.py',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

In [ ]:
import shutil
from pathlib import Path

Path(DRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
if Path(RESULTS_FOLDER).resolve() != Path(DRIVE_RESULTS_DIR).resolve():
    shutil.copytree(RESULTS_FOLDER, DRIVE_RESULTS_DIR, dirs_exist_ok=True)

print('Synced results dir:', DRIVE_RESULTS_DIR)
print('Zip path:', RESULTS_ZIP_PATH)